In [1]:
import random
import math
def f(x,y,z):
    return 7*x + 12*y + 16*z - 4*x*y - 8*x*z - 16*y*z

T = 10

x = 1
y = 1
z = 0

for m in range(50):    
    for n in range(50):
        coin = random.randint(1,3)
        if coin==1:
            D = f(x,y,z) - f(1-x,y,z)
            if D>0:
                x = 1-x
            else:
                p = random.random()
                if p <= math.exp(D/T):
                    x = 1-x
        elif coin==2:
            D = f(x,y,z) - f(x,1-y,z)
            if D>0:
                y = 1-y
            else:
                p = random.random()
                if p <= math.exp(D/T):
                    y = 1-y
        else:
            D = f(x,y,z) - f(x,y,1-z)
            if D>0:
                z = 1-z
            else:
                p = random.random()
                if p <= math.exp(D/T):
                    z = 1-z
    print('m=',m+1,' : ',x,y,z,' : ',f(x,y,z),' : T=',T)
    T*=0.9

m= 1  :  1 0 0  :  7  : T= 10
m= 2  :  0 0 0  :  0  : T= 9.0
m= 3  :  1 1 0  :  15  : T= 8.1
m= 4  :  0 0 0  :  0  : T= 7.29
m= 5  :  0 0 0  :  0  : T= 6.561
m= 6  :  0 0 0  :  0  : T= 5.9049000000000005
m= 7  :  0 0 0  :  0  : T= 5.3144100000000005
m= 8  :  0 0 0  :  0  : T= 4.7829690000000005
m= 9  :  0 0 0  :  0  : T= 4.3046721
m= 10  :  0 0 0  :  0  : T= 3.8742048900000006
m= 11  :  1 1 1  :  7  : T= 3.4867844010000004
m= 12  :  0 0 0  :  0  : T= 3.1381059609000004
m= 13  :  0 0 0  :  0  : T= 2.82429536481
m= 14  :  0 0 0  :  0  : T= 2.541865828329
m= 15  :  0 0 0  :  0  : T= 2.2876792454961
m= 16  :  0 0 0  :  0  : T= 2.05891132094649
m= 17  :  0 0 0  :  0  : T= 1.853020188851841
m= 18  :  1 1 1  :  7  : T= 1.6677181699666568
m= 19  :  1 1 1  :  7  : T= 1.5009463529699911
m= 20  :  1 1 1  :  7  : T= 1.350851717672992
m= 21  :  1 1 1  :  7  : T= 1.2157665459056928
m= 22  :  1 1 1  :  7  : T= 1.0941898913151236
m= 23  :  1 1 1  :  7  : T= 0.9847709021836112
m= 24  :  1 1 1  :  7  : 

In [21]:
import random
import numpy as np

# 1. 目的関数: SI-SDRに基づく損失関数 (最小化が目標)
def calculate_loss(filter_weights, input_signal, target_signal):
    # 実際にはここで s_hat = convolve(input, filter) の後、SI-SDRを計算
    # レポート説明用の仮想損失関数
    prediction = np.dot(filter_weights, input_signal)
    error = np.sum((target_signal - prediction)**2)
    return error 

# 2. 初期設定
T = 100.0  # 初期温度
cooling_rate = 0.98
weights = np.array([0.5, 0.5, 0.5]) # 最適化するフィルタ係数
input_sample = np.array([1.0, 0.8, -0.5])

total_simulate = 2000
target_sample = 1.2
early_stop_buff = [100.0] # 初期値を十分に大きく設定
Val_patiency = 100
count = 0

for m in range(total_simulate):
    # 変数探索: 近傍探索
    idx = random.randint(0, 2)
    old_weight = weights[idx]
    old_loss = calculate_loss(weights, input_sample, target_sample)
    
    # 新しい状態の提案
    adjustment = random.uniform(-0.1, 0.1)
    weights[idx] += adjustment
    new_loss = calculate_loss(weights, input_sample, target_sample)
    
    # エネルギー差の計算 (D)
    D = old_loss - new_loss # 改善された場合は D > 0
    
    if D <= 0: # 性能が悪化した場合は確率的に決定
        p = random.random()
        if p > np.exp(D / T):
            weights[idx] = old_weight # 元の状態に復元
            new_loss = old_loss # Lossを更新前の値に保持
            
    T *= cooling_rate # 冷却

    # アーリーストッピングの判定 (ユーザー様オリジナルのロ직を継承)
    early_stop_buff.append(new_loss)
    
    # 改善が停滞しているかどうかの判定
    # (new_lossが過去の最小値付近で停滞していることを確認)
    if early_stop_buff[m+1] >= min(early_stop_buff[0:m+1]) and new_loss < 0.00001:
        count += 1
    else:
        count = 0

    if count == Val_patiency:
        print(f"loop break_ at loop number = {m+1}, T = {T:.4f}")
        break

print("-" * 30)
print(f"最適化されたフィルタ係数: {weights}")
print(f"最終的な誤差 (Loss): {new_loss:.6f}")

# 目標値との比較確認
final_prediction = np.dot(weights, input_sample)
print(f"最終予測値: {final_prediction:.4f} (目標値: {target_sample})")

loop break_ at loop number = 853, T = 0.0000
------------------------------
最適化されたフィルタ係数: [0.76955539 0.78304508 0.39028362]
最終的な誤差 (Loss): 0.000001
最終予測値: 1.2008 (目標値: 1.2)
